# NB05 v2 — Transformer Fine-Tuning (Multi-Task, Consolidated Labels)

Changes from v1:
- `label_type` consolidated from 89 → 9 canonical classes before encoding
- Enables meaningful multi-task training (binary + abuse_type)

**Requires GPU.** ~3hrs on T4 for 3 models × 3 seeds.

In [5]:
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe


In [6]:
# ── Install dependencies (run once) ────────────────────────────────────────
# VS Code specific: Use the Python interpreter from your active environment
import sys
import subprocess

print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

# Install tokenizers with pre-built wheel (critical for Windows)
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install tokenizers==0.19.1 --only-binary=tokenizers

# Install transformers and other dependencies
!python -m pip install transformers==4.36.0
!python -m pip install datasets accelerate scikit-learn pandas matplotlib seaborn sentencepiece -q

# Verify installations
try:
    import tokenizers
    print(f"✓ tokenizers {tokenizers.__version__}")
    import transformers
    print(f"✓ transformers {transformers.__version__}")
    import torch
    print(f"✓ torch {torch.__version__}")
    print("\n✨ All dependencies installed successfully! ✨")
except ImportError as e:
    print(f"✗ Error: {e}")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python path: c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


  Using cached tokenizers-0.19.1-cp311-none-win_amd64.whl.metadata (6.9 kB)
Using cached tokenizers-0.19.1-cp311-none-win_amd64.whl (2.2 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.36.0 requires tokenizers<0.19,>=0.14, but you have tokenizers 0.19.1 which is incompatible.


  Using cached tokenizers-0.15.2-cp311-none-win_amd64.whl.metadata (6.8 kB)
Using cached tokenizers-0.15.2-cp311-none-win_amd64.whl (2.2 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
✓ tokenizers 0.15.2
✓ transformers 4.36.0
✓ torch 2.11.0+cpu

✨ All dependencies installed successfully! ✨


In [7]:
import os
import json
import time
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    f1_score, accuracy_score, matthews_corrcoef
)

warnings.filterwarnings("ignore")

# ── Hardware / performance setup ────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def infer_num_workers():
    cpu_count = os.cpu_count() or 2
    if os.name == "nt":
        return min(4, max(2, cpu_count // 2))
    return min(6, max(2, cpu_count // 2))

print(f"Suggested num_workers: {infer_num_workers()}")


Device: cpu
Suggested num_workers: 4


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    "models": {
        "banglabert": "csebuetnlp/banglabert",
        "muril": "google/muril-base-cased",
        "xlmr": "xlm-roberta-base",
    },

    # Memory / speed tuned for RTX 4060 Ti 8 GB
    "max_length": 128,
    "batch_size": 16,               # micro-batch
    "eval_batch_size": 32,
    "grad_accum_steps": 2,          # effective batch = 32
    "num_workers": infer_num_workers(),

    # Optimization
    "epochs": 8,
    "encoder_lr": 2e-5,
    "head_lr": 8e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_decay_factor": 0.90,
    "max_grad_norm": 1.0,

    # Loss
    "label_smoothing": 0.03,
    "focal_gamma_binary": 1.5,
    "focal_gamma_abuse_type": 2.5,
    "focal_gamma_severity": 1.5,
    "class_weight_beta": 0.9999,

    # Model regularization
    "dropout": 0.25,
    "head_hidden_dim": 384,

    # Training control
    "patience": 2,
    "use_fp16": torch.cuda.is_available(),
    "use_compile": False,           # keep False for portability
    "seeds": [42, 123, 456],
}

# ── Multi-task head config ──────────────────────────────────────────────────
TASK_CONFIG = {
    "binary": {
        "col": "label_binary",
        "num_classes": 2,
        "loss_weight": 1.0,
        "is_primary": True,
        "monitor_weight": 0.70,
    },
    "abuse_type": {
        "col": "label_type",
        "num_classes": None,
        "loss_weight": 0.90,
        "is_primary": False,
        "monitor_weight": 0.30,
    },
    "severity": {
        "col": "label_severity",
        "num_classes": None,
        "loss_weight": 0.20,
        "is_primary": False,
        "monitor_weight": 0.00,
    },
}

# ── Run control ─────────────────────────────────────────────────────────────
OUTPUT_DIR = "../outputs/models_v2_fix"

# Start with 1 strong verification run. Expand later if needed.
RUN_MODELS = ["banglabert"]
RUN_SEEDS = [42]

FORCE_RETRAIN = False

print(json.dumps(CONFIG, indent=2, default=str))
print(f"\nOUTPUT_DIR : {OUTPUT_DIR}")
print(f"RUN_MODELS  : {RUN_MODELS}")
print(f"RUN_SEEDS   : {RUN_SEEDS}")
print(f"FORCE_RETRAIN: {FORCE_RETRAIN}")


{
  "models": {
    "banglabert": "csebuetnlp/banglabert",
    "muril": "google/muril-base-cased",
    "xlmr": "xlm-roberta-base"
  },
  "max_length": 128,
  "batch_size": 16,
  "eval_batch_size": 32,
  "grad_accum_steps": 2,
  "num_workers": 4,
  "epochs": 8,
  "encoder_lr": 2e-05,
  "head_lr": 8e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "lr_decay_factor": 0.9,
  "max_grad_norm": 1.0,
  "label_smoothing": 0.03,
  "focal_gamma_binary": 1.5,
  "focal_gamma_abuse_type": 2.5,
  "focal_gamma_severity": 1.5,
  "class_weight_beta": 0.9999,
  "dropout": 0.25,
  "head_hidden_dim": 384,
  "patience": 2,
  "use_fp16": false,
  "use_compile": false,
  "seeds": [
    42,
    123,
    456
  ]
}

OUTPUT_DIR : ../outputs/models_v2_fix
RUN_MODELS  : ['banglabert']
RUN_SEEDS   : [42]
FORCE_RETRAIN: False


In [9]:
# ── Load data ──────────────────────────────────────────────────────────────
SPLIT_DIR = "../data/splits"

train_df = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val_df   = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test_df  = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")

TEXT_COL = "text_clean" if "text_clean" in train_df.columns else "text"

for df_ in [train_df, val_df, test_df]:
    df_[TEXT_COL] = df_[TEXT_COL].fillna("").astype(str)

print(f"Text column used: {TEXT_COL}")
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")


Text column used: text_clean
Train: 108,460  Val: 13,557  Test: 13,558


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# LABEL CONSOLIDATION: 89 raw label_type categories → 9 canonical classes
# ══════════════════════════════════════════════════════════════════════════════

TYPE_MAP = {
    # none / not bullying
    'none': 'none', 'not bully': 'none',

    # sexual
    'sexual': 'sexual',

    # threat / call to violence
    'threat': 'threat', 'threat,spam': 'threat',
    'callToViolence': 'threat', 'callToViolence_slander': 'threat',
    'callToViolence_gender': 'threat', 'callToViolence_religion': 'threat',
    'callToViolence_religion_slander': 'threat',
    'callToViolence_gender_religion_slander': 'threat',
    'callToViolence_gender_slander': 'threat',
    'religious,threat': 'threat', 'sexual,threat': 'threat',
    'sexual,religious,threat': 'threat',

    # religious
    'religious': 'religious', 'Religious': 'religious',
    'religion': 'religious', 'religious,spam': 'religious',
    'religion_slander': 'religious',
    'gender_religion': 'religious', 'gender_religion_slander': 'religious',
    'sexual,religious': 'religious',

    # gender
    'gender': 'gender', 'Gender': 'gender', 'gender_slander': 'gender',

    # political
    'Political': 'political',

    # personal
    'Personal Offense': 'personal', 'Body Shaming': 'personal',
    'Origin': 'personal', 'slander': 'personal', 'Misc': 'personal',

    # abusive
    'Abusive/Violence': 'abusive', 'troll': 'abusive',

    # other
    'spam': 'other',
}

PRIORITY = ['threat', 'sexual', 'religious', 'gender', 'political',
            'abusive', 'personal', 'other', 'none']

def consolidate_type(val):
    if not isinstance(val, str) or val.strip() == '':
        return 'none'
    val = val.strip()

    if val in TYPE_MAP:
        return TYPE_MAP[val]

    parts = [v.strip() for v in val.replace(',', '|').replace(';', '|').split('|')]
    candidates = [TYPE_MAP[p] for p in parts if p in TYPE_MAP]
    if candidates:
        for pc in PRIORITY:
            if pc in candidates:
                return pc

    for key, mapped in TYPE_MAP.items():
        if key.lower() in val.lower():
            return mapped

    return 'other'

# Apply consolidation BEFORE building label encoders
for df_ in [train_df, val_df, test_df]:
    if 'label_type' in df_.columns:
        df_['label_type'] = df_['label_type'].apply(consolidate_type)

print("Consolidated label_type classes:")
if 'label_type' in train_df.columns:
    all_types = pd.concat([train_df['label_type'], val_df['label_type'], test_df['label_type']])
    print(sorted(all_types.dropna().unique()))
    print(f"Total consolidated classes: {all_types.dropna().nunique()}")

# ── Auto-detect label classes and create encodings ──────────────────────────
label_encoders = {}
active_tasks = {}

for task_name, task_cfg in TASK_CONFIG.items():
    col = task_cfg["col"]
    if col in train_df.columns:
        all_vals = pd.concat([train_df[col], val_df[col], test_df[col]]).dropna().unique()
        label_map = {v: i for i, v in enumerate(sorted(all_vals))}
        label_encoders[task_name] = label_map
        task_cfg["num_classes"] = len(label_map)
        active_tasks[task_name] = task_cfg
        print(f"Task '{task_name}': {len(label_map)} classes")
    else:
        print(f"Task '{task_name}': column '{col}' not found — SKIPPED")

print(f"\nActive tasks: {list(active_tasks.keys())}")


Consolidated label_type classes:
['abusive', 'gender', 'none', 'other', 'personal', 'political', 'religious', 'sexual', 'threat']
Total consolidated classes: 9
Task 'binary': 2 classes
Task 'abuse_type': 9 classes
Task 'severity': column 'label_severity' not found — SKIPPED

Active tasks: ['binary', 'abuse_type']


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET CLASS + COLLATOR
# ══════════════════════════════════════════════════════════════════════════════

class CyberBullyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders, text_col):
        self.df = df.reset_index(drop=True)
        self.texts = self.df[text_col].fillna("").astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.active_tasks = active_tasks

        self.labels = {}
        for task_name, task_cfg in active_tasks.items():
            col = task_cfg["col"]
            enc = label_encoders[task_name]
            self.labels[task_name] = [
                int(enc.get(v, -1)) for v in self.df[col].fillna("unknown")
            ]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding=False,
        )
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in encoding.items()}
        for task_name in self.labels:
            item[f"label_{task_name}"] = torch.tensor(self.labels[task_name][idx], dtype=torch.long)
        return item


class MultiTaskCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        label_keys = [k for k in features[0].keys() if k.startswith("label_")]
        labels = {k: torch.stack([f[k] for f in features]) for k in label_keys}
        token_features = []
        for f in features:
            token_features.append({k: v for k, v in f.items() if not k.startswith("label_")})

        batch = self.tokenizer.pad(
            token_features,
            padding=True,
            return_tensors="pt"
        )
        batch.update(labels)
        return batch


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# MULTI-TASK TRANSFORMER MODEL
# ══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction="mean", label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.weight,
            reduction="none",
            label_smoothing=self.label_smoothing,
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        if self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


class TaskHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout=0.25, head_hidden_dim=384):
        super().__init__()
        inner_dim = min(head_hidden_dim, hidden_size)
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, inner_dim),
            nn.GELU(),
            nn.LayerNorm(inner_dim),
            nn.Dropout(dropout),
            nn.Linear(inner_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.25, head_hidden_dim=384):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.heads = nn.ModuleDict()
        for task_name, task_cfg in active_tasks.items():
            self.heads[task_name] = TaskHead(
                hidden_size=hidden_size,
                num_classes=task_cfg["num_classes"],
                dropout=dropout,
                head_hidden_dim=head_hidden_dim,
            )

    def _pooled_output(self, outputs, attention_mask):
        last_hidden = outputs.last_hidden_state
        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return 0.5 * cls_vec + 0.5 * mean_vec

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**kwargs)
        pooled = self._pooled_output(outputs, attention_mask)

        logits = {}
        for task_name in self.heads:
            logits[task_name] = self.heads[task_name](pooled)
        return logits


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZER PARAM GROUPS + LOSS HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_layerwise_params(model, encoder_lr, head_lr, decay_factor, weight_decay):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]

    num_layers = 0
    for name, _ in model.encoder.named_parameters():
        for part in name.split("."):
            if part.isdigit():
                num_layers = max(num_layers, int(part) + 1)
    if num_layers == 0:
        num_layers = 12

    param_groups = []

    for name, param in model.encoder.named_parameters():
        if not param.requires_grad:
            continue
        layer_num = 0
        for part in name.split("."):
            if part.isdigit():
                layer_num = int(part)
                break
        lr = encoder_lr * (decay_factor ** (num_layers - layer_num - 1))
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        param_groups.append({"params": [param], "lr": lr, "weight_decay": wd})

    for name, param in model.heads.named_parameters():
        if param.requires_grad:
            wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
            param_groups.append({"params": [param], "lr": head_lr, "weight_decay": wd})

    return param_groups


def build_class_weights(series, label_map, beta=0.9999):
    mapped = series.map(label_map).dropna().astype(int)
    counts = mapped.value_counts().sort_index()
    num_classes = len(label_map)
    weights = np.ones(num_classes, dtype=np.float32)

    for class_idx in range(num_classes):
        count = int(counts.get(class_idx, 0))
        if count > 0:
            effective_num = 1.0 - (beta ** count)
            weights[class_idx] = (1.0 - beta) / max(effective_num, 1e-12)

    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def train_and_evaluate(
    model_key, model_name, train_df, val_df, test_df,
    active_tasks, label_encoders, config, seed
):
    set_seed(seed)

    print(f"\n{'='*70}")
    print(f"Training: {model_key} ({model_name}) | Seed: {seed}")
    print(f"{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collator = MultiTaskCollator(tokenizer)

    train_ds = CyberBullyDataset(train_df, tokenizer, config["max_length"], active_tasks, label_encoders, TEXT_COL)
    val_ds   = CyberBullyDataset(val_df, tokenizer, config["max_length"], active_tasks, label_encoders, TEXT_COL)
    test_ds  = CyberBullyDataset(test_df, tokenizer, config["max_length"], active_tasks, label_encoders, TEXT_COL)

    loader_kwargs = {
        "num_workers": config["num_workers"],
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": config["num_workers"] > 0,
        "collate_fn": collator,
    }

    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config["eval_batch_size"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=config["eval_batch_size"],
        shuffle=False,
        **loader_kwargs,
    )

    model = MultiTaskTransformer(
        model_name,
        active_tasks,
        dropout=config["dropout"],
        head_hidden_dim=config["head_hidden_dim"],
    ).to(device)

    if config.get("use_compile", False) and hasattr(torch, "compile") and device.type == "cuda":
        model = torch.compile(model)

    criteria = {}
    for task_name, task_cfg in active_tasks.items():
        weight_tensor = build_class_weights(
            train_df[task_cfg["col"]],
            label_encoders[task_name],
            beta=config["class_weight_beta"]
        )

        gamma = config.get(f"focal_gamma_{task_name}", 2.0)
        criteria[task_name] = FocalLoss(
            gamma=gamma,
            weight=weight_tensor,
            label_smoothing=config["label_smoothing"],
        )

    optimizer = torch.optim.AdamW(
        get_layerwise_params(
            model,
            encoder_lr=config["encoder_lr"],
            head_lr=config["head_lr"],
            decay_factor=config["lr_decay_factor"],
            weight_decay=config["weight_decay"],
        )
    )

    total_update_steps = math.ceil(len(train_loader) / config["grad_accum_steps"]) * config["epochs"]
    warmup_steps = int(total_update_steps * config["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
    scaler = torch.amp.GradScaler("cuda") if config["use_fp16"] and device.type == "cuda" else None

    best_score = -1.0
    patience_counter = 0
    history = defaultdict(list)
    save_dir = f"{OUTPUT_DIR}/{model_key}_seed{seed}"
    os.makedirs(save_dir, exist_ok=True)

    with open(f"{save_dir}/label_encoders.json", "w", encoding="utf-8") as f:
        json.dump(label_encoders, f, ensure_ascii=False, indent=2)

    for epoch in range(config["epochs"]):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        start_time = time.time()

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.autocast(device_type=device.type, enabled=config["use_fp16"] and device.type == "cuda"):
                logits = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    token_type_ids=batch.get("token_type_ids")
                )

                loss = 0.0
                for task_name, task_cfg in active_tasks.items():
                    task_labels = batch[f"label_{task_name}"]
                    valid_mask = task_labels >= 0
                    if valid_mask.any():
                        task_loss = criteria[task_name](
                            logits[task_name][valid_mask],
                            task_labels[valid_mask]
                        )
                        loss = loss + task_cfg["loss_weight"] * task_loss

                loss = loss / config["grad_accum_steps"]

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if step % config["grad_accum_steps"] == 0 or step == len(train_loader):
                if scaler is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item() * config["grad_accum_steps"]

        avg_loss = total_loss / max(len(train_loader), 1)
        history["train_loss"].append(avg_loss)

        val_metrics = evaluate_tasks(model, val_loader, active_tasks, label_encoders)
        binary_f1 = val_metrics.get("binary", {}).get("macro_f1", 0.0)
        abuse_f1 = val_metrics.get("abuse_type", {}).get("macro_f1", 0.0)
        severity_f1 = val_metrics.get("severity", {}).get("macro_f1", 0.0)

        monitor_score = (
            active_tasks.get("binary", {}).get("monitor_weight", 0.0) * binary_f1 +
            active_tasks.get("abuse_type", {}).get("monitor_weight", 0.0) * abuse_f1 +
            active_tasks.get("severity", {}).get("monitor_weight", 0.0) * severity_f1
        )
        history["val_monitor_score"].append(monitor_score)
        history["val_binary_f1"].append(binary_f1)
        history["val_abuse_type_f1"].append(abuse_f1)

        elapsed = time.time() - start_time
        status = ""
        if monitor_score > best_score:
            best_score = monitor_score
            torch.save(model.state_dict(), f"{save_dir}/best_model.pt")
            patience_counter = 0
            status = " ✅ BEST"
        else:
            patience_counter += 1

        print(
            f"  Epoch {epoch+1:2d}/{config['epochs']} | "
            f"Loss: {avg_loss:.4f} | "
            f"Val binary F1: {binary_f1:.4f} | "
            f"Val abuse F1: {abuse_f1:.4f} | "
            f"Monitor: {monitor_score:.4f} | "
            f"Time: {elapsed/60:.1f} min{status}"
        )

        if patience_counter >= config["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    state_dict = torch.load(f"{save_dir}/best_model.pt", map_location=device, weights_only=True)
    model.load_state_dict(state_dict)

    test_metrics = evaluate_tasks(model, test_loader, active_tasks, label_encoders)
    test_logits = get_logits(model, test_loader, active_tasks)
    val_logits = get_logits(model, val_loader, active_tasks)

    result = {
        "model": model_key,
        "seed": seed,
        "best_val_monitor_score": round(best_score, 4),
        "test_metrics": test_metrics,
        "history": {k: [round(v, 4) for v in vals] for k, vals in history.items()}
    }

    with open(f"{save_dir}/results.json", "w") as f:
        json.dump(result, f, indent=2)

    torch.save(test_logits, f"{save_dir}/test_logits.pt")
    torch.save(val_logits, f"{save_dir}/val_logits.pt")

    print(f"\n  ✅ Results saved to {save_dir}")
    return result


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION & LOGIT EXTRACTION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_tasks(model, dataloader, active_tasks, label_encoders):
    model.eval()
    all_preds = {t: [] for t in active_tasks}
    all_labels = {t: [] for t in active_tasks}

    for batch in dataloader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids")
        )

        for task_name in active_tasks:
            task_labels = batch[f"label_{task_name}"].detach().cpu().numpy()
            task_logits = logits[task_name].detach().cpu()
            task_preds = task_logits.argmax(dim=-1).numpy()

            valid = task_labels >= 0
            all_preds[task_name].extend(task_preds[valid])
            all_labels[task_name].extend(task_labels[valid])

    metrics = {}
    for task_name in active_tasks:
        y_true = np.array(all_labels[task_name])
        y_pred = np.array(all_preds[task_name])

        metrics[task_name] = {
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
            "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "mcc": round(matthews_corrcoef(y_true, y_pred), 4),
        }

    return metrics


@torch.no_grad()
def get_logits(model, dataloader, active_tasks):
    model.eval()
    all_logits = {t: [] for t in active_tasks}

    for batch in dataloader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids")
        )
        for task_name in active_tasks:
            all_logits[task_name].append(logits[task_name].detach().cpu())

    return {t: torch.cat(v, dim=0) for t, v in all_logits.items()}


## Run Training for All Models × Seeds

This is the main training loop. Each model is trained 3 times (different seeds) for stability.

In [16]:
import os
import json
import torch

def is_valid_model(save_dir):
    model_path = f"{save_dir}/best_model.pt"
    if not os.path.exists(model_path):
        return False
    try:
        torch.load(model_path, map_location="cpu", weights_only=True)
        return True
    except Exception as e:
        print(f"  ⚠️ Corrupted model file at {model_path}: {e}")
        return False

all_results = []
os.makedirs(OUTPUT_DIR, exist_ok=True)

selected_models = {k: v for k, v in CONFIG["models"].items() if k in RUN_MODELS}

for model_key, model_name in selected_models.items():
    for seed in RUN_SEEDS:
        save_dir = f"{OUTPUT_DIR}/{model_key}_seed{seed}"

        if (not FORCE_RETRAIN) and is_valid_model(save_dir):
            print(f"⏭️ Skipping {model_key} seed={seed} – already completed and valid")
            results_path = f"{save_dir}/results.json"
            if os.path.exists(results_path):
                with open(results_path, "r") as f:
                    existing_result = json.load(f)
                    all_results.append(existing_result)
            continue
        elif os.path.exists(f"{save_dir}/best_model.pt"):
            print(f"🔄 Re-training {model_key} seed={seed} – existing model file is corrupted/incomplete")
        else:
            print(f"🆕 Starting {model_key} seed={seed} – no existing model found")

        try:
            result = train_and_evaluate(
                model_key, model_name,
                train_df, val_df, test_df,
                active_tasks, label_encoders,
                CONFIG, seed
            )
            all_results.append(result)
        except Exception as e:
            print(f"\n❌ FAILED: {model_key} seed={seed}: {e}")
            import traceback
            traceback.print_exc()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n\n{'='*70}")
print(f"TRAINING COMPLETE: {len(all_results)} runs finished (including resumed)")
print(f"{'='*70}")


🆕 Starting banglabert seed=42 – no existing model found

Training: banglabert (csebuetnlp/banglabert) | Seed: 42

❌ FAILED: banglabert seed=42: keys must be str, int, float, bool or None, not int64


TRAINING COMPLETE: 0 runs finished (including resumed)


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_2924\2200959361.py", line 39, in <module>
    result = train_and_evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_2924\2802565943.py", line 95, in train_and_evaluate
    json.dump(label_encoders, f, ensure_ascii=False, indent=2)
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\json\__init__.py", line 179, in dump
    for chunk in iterable:
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\json\encoder.py", line 432, in _iterencode
    yield from _iterencode_dict(o, _current_indent_level)
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\json\encoder.py", line 406, in _iterencode_dict
    yield from chunks
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\json\encoder.py", line 377, in _iterencode_dict
    raise TypeError(f'keys must be str, int, float, bool or None, '
TypeError: keys must be s

In [17]:
if len(all_results) == 0:
    print("⚠️ No results to display. Training failed for all runs.")
else:
    summary_rows = []
    for r in all_results:
        row = {
            "model": r["model"],
            "seed": r["seed"],
            "best_val_monitor_score": r.get("best_val_monitor_score", np.nan),
        }
        for task_name, task_metrics in r["test_metrics"].items():
            for metric_name, metric_val in task_metrics.items():
                row[f"{task_name}_{metric_name}"] = metric_val
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

    print("\n── Averaged across seeds (mean ± std) ──")
    numeric_cols = [c for c in summary_df.columns if c not in ["model", "seed"]]
    avg_df = summary_df.groupby("model")[numeric_cols].agg(["mean", "std"]).round(4)
    print(avg_df.to_string())

    summary_df.to_csv(f"{OUTPUT_DIR}/transformer_results_all.csv", index=False)
    avg_df.to_csv(f"{OUTPUT_DIR}/transformer_results_averaged.csv")
    print(f"\n✅ Saved results to {OUTPUT_DIR}")

    preferred_plot = "abuse_type_macro_f1" if "abuse_type_macro_f1" in summary_df.columns else "binary_macro_f1"
    if preferred_plot in summary_df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        models = summary_df["model"].unique()
        x = np.arange(len(models))

        means = [summary_df[summary_df["model"] == m][preferred_plot].mean() for m in models]
        stds  = [summary_df[summary_df["model"] == m][preferred_plot].std() for m in models]

        bars = ax.bar(x, means, yerr=stds, capsize=5,
                      color=["#3498db", "#e74c3c", "#2ecc71"][:len(models)],
                      edgecolor="black", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.set_ylabel(preferred_plot.replace("_", " ").title())
        ax.set_title(f"Transformer Models — {preferred_plot.replace('_', ' ').title()} (mean ± std)", fontweight="bold")
        ax.set_ylim(0, 1.0)

        for bar, m, s in zip(bars, means, stds):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0 if pd.isna(s) else s) + 0.01,
                f"{m:.3f}±{0.0 if pd.isna(s) else s:.3f}",
                ha="center", fontsize=10, fontweight="bold"
            )

        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/transformer_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("⚠️ No matching metric column found for plotting.")


⚠️ No results to display. Training failed for all runs.


In [18]:
# ── Visualization (standalone cell — safe even if training failed) ─────────
if 'all_results' not in dir() or len(all_results) == 0:
    print("⚠️ No results available. Run the training cell first.")
elif 'summary_df' not in dir():
    print("⚠️ summary_df not found. Re-run the results cell above.")
elif len(summary_df) > 0 and "binary_macro_f1" in summary_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    models = summary_df["model"].unique()
    x = np.arange(len(models))
    
    means = [summary_df[summary_df["model"]==m]["binary_macro_f1"].mean() for m in models]
    stds  = [summary_df[summary_df["model"]==m]["binary_macro_f1"].std() for m in models]
    
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=["#3498db", "#e74c3c", "#2ecc71"][:len(models)],
                  edgecolor="black", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel("Macro-F1 (Binary)")
    ax.set_title("Transformer Models — Test Macro-F1 (mean ± std across seeds)", fontweight="bold")
    ax.set_ylim(0, 1.0)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
                f"{m:.3f}±{s:.3f}", ha="center", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/models/transformer_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠️ 'binary_macro_f1' not found in results – skipping plot.")


⚠️ No results available. Run the training cell first.


---
**Next:** Notebook `06_ensemble_and_threshold.ipynb` — Build weighted logits ensemble from the saved logits, tune thresholds, and evaluate final system.